# Simulator Brier Comparison

This notebook lists every exact contract/question template the bundled simulator can benchmark, then compares three rules on identical historical rows:

- `simulator`: frozen simulator probability
- `always_50`: always predict 0.5
- `empirical_rate`: always predict the exact-contract empirical YES rate available before the evaluation set

Scopes: `all_history` uses rolling-origin out-of-sample rows; `wc2026` uses the tracked WC2026 simulator replay rows with pre-2026 empirical rates.

In [ ]:
from pathlib import Path
import csv
import gzip
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "simulator/data/processed/simulation_evidence.json").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.build_simulator_family_benchmarks import (
    DEFAULT_SOURCE_ROOT,
    _canonical_key,
    family_from_contract,
    load_comparison_rows,
)
from bot.simulator_contracts import observation_unit, questions_for_contract
from bot.wc2026_evidence import supports_contract

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 200)

ARTIFACT_PATH = ROOT / "simulator/data/processed/simulation_evidence.json"
WC2026_ROWS = DEFAULT_SOURCE_ROOT / "exports/wc2026_simulator_oos_rows.csv.gz"

artifact = json.loads(ARTIFACT_PATH.read_text())
contracts = artifact.get("contracts") or {}

def empirical_prior(key):
    row = ((contracts.get(key) or {}).get("empirical_rate") or {}).get("all_history") or {}
    if row.get("available") and row.get("rate") is not None:
        return float(row["rate"])
    return None


In [ ]:
supported_records = []
for key in sorted(contracts):
    questions = questions_for_contract(key, "HOME", "AWAY")
    if not questions or not supports_contract(key):
        continue
    supported_records.append({
        "family": family_from_contract(key),
        "contract_key": key,
        "observation_unit": observation_unit(key),
        "questions_per_match": len(questions),
        "question_templates": " | ".join(questions),
        "empirical_rate_all_history": empirical_prior(key),
    })

supported = pd.DataFrame(supported_records)
contract_keys = set(supported.contract_key)

print(f"Supported benchmarkable contracts: {len(supported)}")
print(f"Supported families: {supported.family.nunique()}")
supported.sort_values(["family", "contract_key"])


In [ ]:
all_history_rows = load_comparison_rows(DEFAULT_SOURCE_ROOT, contract_keys)

def read_wc2026_rows():
    rows = []
    with gzip.open(WC2026_ROWS, "rt", newline="") as handle:
        for row in csv.DictReader(handle):
            key = _canonical_key(row["contract_key"])
            if key not in contract_keys:
                continue
            p_empirical = empirical_prior(key)
            if p_empirical is None:
                continue
            rows.append({
                "family": family_from_contract(key),
                "contract_key": key,
                "match_id": row["match_id"],
                "outcome": float(row["outcome"]),
                "p_model": float(row["p_model"]),
                "p_empirical": p_empirical,
            })
    return rows

wc2026_rows = read_wc2026_rows()

def brier_summary(rows, scope, by=()):
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame()
    df = df[df["p_empirical"].notna()].copy()
    df["simulator"] = (df["p_model"] - df["outcome"]) ** 2
    df["always_50"] = (0.5 - df["outcome"]) ** 2
    df["empirical_rate"] = (df["p_empirical"] - df["outcome"]) ** 2

    if by:
        out = df.groupby(list(by), dropna=False).agg(
            observations=("outcome", "size"),
            matches=("match_id", "nunique"),
            contracts=("contract_key", "nunique"),
            simulator=("simulator", "mean"),
            always_50=("always_50", "mean"),
            empirical_rate=("empirical_rate", "mean"),
        ).reset_index()
    else:
        out = pd.DataFrame([{
            "observations": len(df),
            "matches": df["match_id"].nunique(),
            "contracts": df["contract_key"].nunique(),
            "simulator": df["simulator"].mean(),
            "always_50": df["always_50"].mean(),
            "empirical_rate": df["empirical_rate"].mean(),
        }])

    out.insert(0, "scope", scope)
    out["sim_minus_50"] = out["simulator"] - out["always_50"]
    out["sim_minus_empirical"] = out["simulator"] - out["empirical_rate"]
    metric_cols = ["simulator", "always_50", "empirical_rate", "sim_minus_50", "sim_minus_empirical"]
    out[metric_cols] = out[metric_cols].round(6)
    return out.sort_values(["scope", *list(by)]).reset_index(drop=True)

print(f"all_history comparable rows: {len(all_history_rows):,}")
print(f"wc2026 comparable rows: {len(wc2026_rows):,}")


In [ ]:
overall_brier = pd.concat([
    brier_summary(all_history_rows, "all_history"),
    brier_summary(wc2026_rows, "wc2026"),
], ignore_index=True)

family_brier = pd.concat([
    brier_summary(all_history_rows, "all_history", ("family",)),
    brier_summary(wc2026_rows, "wc2026", ("family",)),
], ignore_index=True)

contract_brier = pd.concat([
    brier_summary(all_history_rows, "all_history", ("family", "contract_key")),
    brier_summary(wc2026_rows, "wc2026", ("family", "contract_key")),
], ignore_index=True)

overall_brier


In [ ]:
family_brier.sort_values(["scope", "sim_minus_empirical", "family"])


In [ ]:
contract_brier.sort_values(["scope", "sim_minus_empirical", "family", "contract_key"])


In [ ]:
ax = overall_brier.set_index("scope")[["simulator", "always_50", "empirical_rate"]].plot(
    kind="bar", figsize=(7, 4), rot=0, color=["#2f6f9f", "#9a9a9a", "#b85c38"],
)
ax.set_title("Overall Brier Score by Evaluation Scope")
ax.set_ylabel("Brier score")
ax.set_xlabel("")
ax.legend(title="Rule", frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()


In [ ]:
scopes = list(family_brier["scope"].drop_duplicates())
fig, axes = plt.subplots(1, len(scopes), figsize=(12, max(4, 0.32 * family_brier["family"].nunique())), sharex=True)
if len(scopes) == 1:
    axes = [axes]
for ax, scope in zip(axes, scopes, strict=True):
    sub = family_brier[family_brier["scope"].eq(scope)].sort_values("sim_minus_empirical")
    colors = ["#2f6f9f" if value <= 0 else "#b85c38" for value in sub["sim_minus_empirical"]]
    ax.barh(sub["family"], sub["sim_minus_empirical"], color=colors)
    ax.axvline(0, color="#333333", linewidth=1)
    ax.set_title(scope)
    ax.set_xlabel("Simulator Brier - empirical-rate Brier")
    ax.grid(axis="x", alpha=0.25)
fig.suptitle("Family-Level Delta vs Empirical-Rate Baseline", y=1.02)
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, len(scopes), figsize=(11, 4), sharex=True, sharey=True)
if len(scopes) == 1:
    axes = [axes]
for ax, scope in zip(axes, scopes, strict=True):
    sub = contract_brier[contract_brier["scope"].eq(scope)].copy()
    sizes = 20 + 180 * (sub["observations"] / sub["observations"].max())
    ax.scatter(sub["empirical_rate"], sub["simulator"], s=sizes, alpha=0.65, color="#2f6f9f", edgecolor="white", linewidth=0.6)
    lo = min(sub["empirical_rate"].min(), sub["simulator"].min(), 0.0)
    hi = max(sub["empirical_rate"].max(), sub["simulator"].max(), 0.3)
    ax.plot([lo, hi], [lo, hi], color="#555555", linestyle="--", linewidth=1)
    ax.set_title(scope)
    ax.set_xlabel("Empirical-rate Brier")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Simulator Brier")
fig.suptitle("Contract-Level Brier: Simulator vs Empirical-Rate Baseline", y=1.03)
plt.tight_layout()
